In [ ]:
# ── CV Embedding -> ChromaDB (fondasi scoring berbasis job desc) ──────
# Jalankan SETELAH ekstraksi (cv_semantic_chunking_v1) mengisi data/processed/*.json
import ollama
import json
import chromadb
from pathlib import Path

EMBED_MODEL   = "qwen3-embedding:0.6b"      # multilingual, ~1.2GB, kuat di kelasnya
PROCESSED_DIR = Path("../data/processed")   # JSON hasil ekstraksi
CHROMA_DIR    = Path("../data/chroma")      # tempat vektor disimpan (persisten)
COLLECTION    = "cv_chunks"

In [ ]:
# ── 1) Embedding via Ollama: bedakan DOCUMENT vs QUERY ───────────────
# Qwen3-Embedding pakai "asymmetric search": query diberi prefix instruksi,
# document di-embed polos. Pakai prefix yang benar = retrieval lebih akurat.
QUERY_TASK = "Given a job requirement, retrieve the candidate experience, skills, or education that best matches it"

def embed_document(text: str) -> list:
    return ollama.embed(model=EMBED_MODEL, input=text)["embeddings"][0]

def embed_query(text: str) -> list:
    prompt = f"Instruct: {QUERY_TASK}\nQuery:{text}"
    return ollama.embed(model=EMBED_MODEL, input=prompt)["embeddings"][0]

_dim = len(embed_document("test"))
print(f"Model: {EMBED_MODEL} | dimensi embedding: {_dim}")

In [3]:
# ── 2) Pecah 1 CV JSON jadi beberapa chunk + metadata (field-aware) ──
# Tiap chunk = 1 unit semantik (summary / 1 experience / skills / dst),
# disimpan dengan metadata supaya nanti bisa difilter saat scoring.
def cv_to_chunks(cv: dict, cv_id: str) -> list:
    chunks = []
    name = cv.get("personal_info", {}).get("name", "")

    def add(kind, text, **meta):
        text = (text or "").strip()
        if text:
            chunks.append({
                "id": f"{cv_id}::{kind}::{len(chunks)}",
                "text": text,
                "meta": {"cv_id": cv_id, "name": name, "type": kind, **meta},
            })

    add("summary", cv.get("summary", ""))

    sk = cv.get("skills", {})
    all_skills = sk.get("hard_skills", []) + sk.get("soft_skills", [])
    add("skills", "Skills: " + ", ".join(all_skills), skills=", ".join(all_skills))

    for e in cv.get("experience", []):
        ks = ", ".join(e.get("key_skills", []))
        txt = f'{e.get("role","")} at {e.get("company","")}. {e.get("summary","")} Skills: {ks}'
        add("experience", txt,
            company=e.get("company", ""), role=e.get("role", ""),
            start_date=e.get("start_date", ""), end_date=e.get("end_date", ""),
            is_current=bool(e.get("is_current", False)), key_skills=ks)

    for ed in cv.get("education", []):
        txt = f'{ed.get("degree","")} in {ed.get("field_of_study","")} at {ed.get("institution","")}'
        add("education", txt, institution=ed.get("institution", ""))

    for c in cv.get("certifications", []):
        add("certification", f'{c.get("name","")} - {c.get("issuer","")}', issuer=c.get("issuer", ""))

    if cv.get("achievements"):
        add("achievements", " ".join(cv["achievements"]))

    return chunks

In [ ]:
# ── 3) Bangun semua chunk -> embed -> simpan ke ChromaDB ─────────────
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:                                  # reset biar idempotent kalau di-run ulang
    client.delete_collection(COLLECTION)
except Exception:
    pass
col = client.create_collection(COLLECTION, metadata={"hnsw:space": "cosine"})

all_chunks = []
files = sorted(PROCESSED_DIR.glob("*.json"))
for jf in files:
    cv = json.load(open(jf, encoding="utf-8"))
    all_chunks += cv_to_chunks(cv, jf.stem)

print(f"{len(all_chunks)} chunk dari {len(files)} CV. Meng-embed (sequential)...")
embeddings = [embed_document(c["text"]) for c in all_chunks]

col.add(
    ids=[c["id"] for c in all_chunks],
    documents=[c["text"] for c in all_chunks],
    embeddings=embeddings,
    metadatas=[c["meta"] for c in all_chunks],
)
print(f"Selesai. {col.count()} vektor tersimpan di {CHROMA_DIR}/")

In [5]:
# ── 4) Cari chunk CV paling cocok dengan sebuah requirement (preview) ─
def search(requirement: str, n: int = 5, where: dict = None):
    qe = embed_query(requirement)
    res = col.query(query_embeddings=[qe], n_results=n, where=where)
    print(f"QUERY: {requirement}\n")
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f'[{1 - dist:.3f}] {meta.get("name","?")} | {meta["type"]:11} | {doc[:90]}')

# contoh requirement
search("butuh pengalaman backend Java Spring Boot dan microservices")
# search("perlu kandidat lulusan data science")
# contoh dengan filter metadata (cuma chunk experience):
# search("memimpin tim penjualan regional", where={"type": "experience"})

QUERY: butuh pengalaman backend Java Spring Boot dan microservices

[0.702] RIZA ALAMSYA | summary     | Results-driven Software Engineer with over 5 years of experience in designing and implemen
[0.671] RIZA ALAMSYA | experience  | Senior Software Engineer at Agoh Marketing Sdn. Bhd. Built Agoh Management System from scr
[0.635] RIZA ALAMSYA | skills      | Skills: Java, Spring Boot, Java EE, PHP (Laravel), Microservices Architecture, Event-Drive
[0.596] RIZA ALAMSYA | experience  | Fullstack Developer at Collega by Telkom Indonesia. Developed Guest Book application using
[0.582] RIZA ALAMSYA | experience  | Senior Java Developer at Puncak Tegap Sdn. Bhd. Maintained and enhanced the eTanah project
